# 02 – Baseline CNN (from scratch)

3 conv blocks (32/64/128) → GAP → Dropout → Linear(1)

Input: native 32×32 · BCEWithLogitsLoss · Adam 1e-3 · ~10 epochs

In [ ]:
# --- Colab setup (run first) ---
# Runtime → Change runtime type → T4 GPU
!pip -q install kagglehub grad-cam

from pathlib import Path
from google.colab import drive
import kagglehub

# Everything we produce goes to Drive so it survives Colab restarts
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/ai-image-detector")
MODELS = OUT / "models"; REPORT = OUT / "report"
MODELS.mkdir(parents=True, exist_ok=True); REPORT.mkdir(parents=True, exist_ok=True)

# Downloads CIFAKE once (~100 MB); contains train/{REAL,FAKE} and test/{REAL,FAKE}
DATA_ROOT = Path(kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images"))
print("Data:", DATA_ROOT, "| Outputs:", OUT)

In [ ]:
import os, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

BATCH = 128
EPOCHS = 10
LR = 1e-3

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

full_train = datasets.ImageFolder(DATA_ROOT / "train", transform=transform)
# ImageFolder sorts alphabetically → FAKE=0, REAL=1. We want AI=1, REAL=0
# So we invert labels later or remap class_to_idx
print("Classes:", full_train.class_to_idx)  # expect {'FAKE': 0, 'REAL': 1}

# 90/10 train/val split
n_val = int(0.1 * len(full_train))
n_train = len(full_train) - n_val
train_ds, val_ds = random_split(full_train, [n_train, n_val],
                                generator=torch.Generator().manual_seed(SEED))

test_ds = datasets.ImageFolder(DATA_ROOT / "test", transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)
print(f"Train {n_train:,} | Val {n_val:,} | Test {len(test_ds):,}")

In [ ]:
class BaselineCNN(nn.Module):
    """3 conv blocks → GAP → Dropout → Linear(1)"""
    def __init__(self):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            block(3, 32),
            block(32, 64),
            block(64, 128),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)

model = BaselineCNN().to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, preds, targets = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.float().to(DEVICE)
        # ImageFolder: FAKE=0, REAL=1 → we want AI=1 so invert
        y = 1.0 - y
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * x.size(0)
        preds.append(torch.sigmoid(logits).detach().cpu())
        targets.append(y.cpu())
    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    acc = accuracy_score(targets, preds >= 0.5)
    return total_loss / len(loader.dataset), acc, preds, targets

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, train=True)
    va_loss, va_acc, _, _ = run_epoch(val_loader, train=False)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(va_acc)
    print(f"Epoch {epoch:02d} | train {tr_loss:.4f}/{tr_acc:.3f} | val {va_loss:.4f}/{va_acc:.3f} | {time.time()-t0:.1f}s")
    if va_loss < best_val:
        best_val = va_loss
        torch.save(model.state_dict(), MODELS / "baseline_cnn_best.pt")
        print("  → saved best")

In [ ]:
# Test metrics
model.load_state_dict(torch.load(MODELS / "baseline_cnn_best.pt", map_location=DEVICE))
_, test_acc, preds, targets = run_epoch(test_loader, train=False)
prec, rec, f1, _ = precision_recall_fscore_support(targets, preds >= 0.5, average="binary")
auc = roc_auc_score(targets, preds)
cm = confusion_matrix(targets, preds >= 0.5)

print(f"Test Acc  {test_acc:.4f}")
print(f"Precision {prec:.4f}  Recall {rec:.4f}  F1 {f1:.4f}  AUC {auc:.4f}")
print("Confusion matrix (rows=true, cols=pred):\n", cm)

# Curves
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout()
plt.savefig(REPORT / "baseline_curves.png", dpi=150)
plt.show()